# Heaps & Priority Queues

Notes and runnable examples on the **binary heap** and Python's `heapq` — how a priority queue gets O(log n) push/pop and an O(n) build, all on top of a plain `list`.

**Contents**

1. **What it is** — the heap invariant, priority queues
2. **CPython internals** — a binary heap living inside a `list`
3. **Push & pop** — sift up / sift down
4. **Building is O(n)** — `heapify` vs n pushes
5. **Priority queues in practice** — tuples, tie-breakers, max-heaps
6. **Two heaps** — the running median
7. **When to use what**
8. **Complexity cheat-sheet**

## 1. What it is

A **heap** is a tree that keeps its most-extreme element instantly reachable. The defining **heap invariant** is that every parent compares favorably to its children — `parent <= children` for a *min-heap*, `parent >= children` for a *max-heap*. That makes it the natural backing for a **priority queue**: always serve the most-urgent item next, with O(1) peek at the extreme and O(log n) to insert or remove it.

Python's `heapq` implements a **min-heap** — the smallest element is always at the front. Note a heap is only *partially* ordered: it knows the minimum, but the rest is not sorted.

## 2. CPython internals — a binary heap inside a `list`

`heapq` is unusual: there is **no heap object**. It's a module of functions that operate on an ordinary `list`, treating it as a **complete binary tree** laid out level by level. That lets parent/child links be pure index arithmetic — no pointers, no node objects:

```
node i  ->  children at  2*i + 1  and  2*i + 2
node i  ->  parent   at  (i - 1) // 2
```

Because the tree is always *complete* (filled left-to-right, no gaps), the list is dense and indexing is O(1). The functions are written in C — `heapq` does `from _heapq import *`, falling back to the pure-Python implementation only if the C module is unavailable. So the "heap" you build is just a list whose elements happen to satisfy the invariant:

In [1]:
import heapq

h = []
for x in [5, 3, 8, 1, 9, 2, 7]:
    heapq.heappush(h, x)

print("underlying list :", h)          # heap-ordered, NOT fully sorted
print("min at h[0]     :", h[0])

invariant = all(h[i] <= h[c]
                for i in range(len(h))
                for c in (2*i + 1, 2*i + 2) if c < len(h))
print("parent <= children everywhere:", invariant)
print("implemented in  :", heapq.heappush.__module__, "(C accelerator)")


underlying list : [1, 3, 2, 5, 9, 8, 7]
min at h[0]     : 1
parent <= children everywhere: True
implemented in  : _heapq (C accelerator)


## 3. Push & pop — sift up / sift down

Both operations restore the invariant by moving a single element along one root-to-leaf path, which is why each is **O(log n)** (the tree's height):

- **`heappush`** appends the value at the end (the next free leaf), then **sifts it up**: swap with its parent while it's smaller.
- **`heappop`** returns `h[0]` (the min), moves the *last* element into the root hole, then **sifts it down**: swap with the smaller child until the invariant holds again.

A consequence: repeatedly popping yields elements in **sorted order** — that's heapsort.

> Insider note: CPython's internal helpers are named `_siftup` / `_siftdown` in a way that looks *backwards* versus most textbooks (`_siftup` actually moves a hole downward). The public `heappush`/`heappop` behavior is exactly as described above.

In [2]:
import heapq

h = [5, 3, 8, 1, 9, 2, 7]
heapq.heapify(h)
print("drain via heappop:", [heapq.heappop(h) for _ in range(len(h))])   # sorted ascending


drain via heappop: [1, 2, 3, 5, 7, 8, 9]


## 4. Building a heap is O(n), not O(n log n)

Building by pushing n items one at a time costs n × O(log n) = **O(n log n)**. But `heapify` does better: it arranges the list in place from the **bottom up** (Floyd's build-heap), sifting down each internal node. The leaves — about half the array — need no work, and nodes near the bottom sift down only a short distance. Summing the cost over all levels gives a converging series that totals **O(n)**.

So `heapify` is both asymptotically better *and* a smaller constant than repeated pushes. Watch the per-element cost: it stays flat for `heapify` (linear), and `heapify` beats n pushes at every size:

In [3]:
import heapq, timeit, random

random.seed(0)
print(f"{'N':>8}{'heapify':>11}{'ns/elem':>9}   {'N×push':>11}{'ns/elem':>9}")
for N in (100_000, 200_000, 400_000):
    data = [random.random() for _ in range(N)]
    th = timeit.timeit(lambda: heapq.heapify(data[:]), number=1)

    def pushes():
        h = []
        for v in data:
            heapq.heappush(h, v)
    tp = timeit.timeit(pushes, number=1)

    print(f"{N:>8}{th*1000:>9.1f}ms{th/N*1e9:>9.1f}   {tp*1000:>9.1f}ms{tp/N*1e9:>9.1f}")


       N    heapify  ns/elem        N×push  ns/elem
  100000      3.8ms     38.5         7.0ms     69.6
  200000      7.2ms     35.8        13.3ms     66.6
  400000     15.1ms     37.7        29.9ms     74.6


## 5. Priority queues in practice

The usual pattern is to push `(priority, item)` tuples — the heap orders by the first field. Two refinements matter in real code:

1. Push `(priority, tiebreaker, item)` with a unique, ever-increasing **counter** as the tiebreaker. It gives **FIFO order among equal priorities**, *and* it stops Python from trying to compare the `item`s (which may be unorderable) when two priorities tie.
2. `heapq` is **min-only**. For a max-heap, **negate** the priority (or reach for the private `_max` helpers).

In [4]:
import heapq, itertools

pq = []
counter = itertools.count()          # unique, ever-increasing tiebreaker

def add_task(priority, task):
    heapq.heappush(pq, (priority, next(counter), task))

add_task(2, "write report")
add_task(1, "fix prod bug")
add_task(2, "reply email")           # same priority as the report, but queued later
add_task(1, "deploy")

while pq:
    priority, _, task = heapq.heappop(pq)
    print(f"  p{priority}  {task}")


  p1  fix prod bug
  p1  deploy
  p2  write report
  p2  reply email


### Handy helpers

`heapq` ships a few one-shot tools worth knowing:

- `nsmallest(k, it)` / `nlargest(k, it)` — a partial sort in **O(n log k)**, better than sorting when k ≪ n.
- `heappushpop(h, x)` / `heapreplace(h, x)` — push and pop in a **single** sift, cheaper than doing both.
- `merge(*sorted_iterables)` — lazily merge already-sorted inputs.

In [5]:
import heapq

nums = [5, 3, 8, 1, 9, 2, 7, 4, 6]
print("nsmallest(3):", heapq.nsmallest(3, nums))
print("nlargest(3) :", heapq.nlargest(3, nums))

# max-heap by negation
maxh = []
for x in [5, 3, 8, 1]:
    heapq.heappush(maxh, -x)
print("max via negation:", -maxh[0])

# push-then-pop in one efficient step
h = [1, 3, 5]
print("heappushpop([1,3,5], 4):", heapq.heappushpop(h, 4), "-> heap now", h)


nsmallest(3): [1, 2, 3]
nlargest(3) : [9, 8, 7]
max via negation: 8
heappushpop([1,3,5], 4): 1 -> heap now [3, 4, 5]


## 6. Two heaps — the running median

The median of a stream is awkward: there's no single extreme to track, it lives in the *middle*. The trick is to split the data across **two heaps** that meet in the middle:

- a **max-heap** holding the smaller half (its largest element sits on top, right at the divide), and
- a **min-heap** holding the larger half (its smallest element sits on top).

`heapq` is min-only, so the max-heap is the usual **negation** trick from the priority-queue section. Keep the two sizes balanced — never differing by more than one — and the median is always one or two peeks away:

- sizes equal → average the two tops (the value straddles the gap);
- one heap larger → its top *is* the median.

Each `add` is two O(log n) heap ops (push, then a rebalance push/pop), and `median` is O(1) peeks. That beats re-sorting the whole prefix (O(n log n)) on every query.

In [6]:
import heapq, random, statistics


class MedianFinder:
    """Streaming median via two balanced heaps."""

    def __init__(self):
        self.lo = []   # max-heap (negated) of the smaller half
        self.hi = []   # min-heap of the larger half

    def add(self, x):
        # Route x to the correct half so every lo element stays <= every hi element.
        if self.lo and x < -self.lo[0]:
            heapq.heappush(self.lo, -x)
        else:
            heapq.heappush(self.hi, x)
        # Rebalance so |lo| - |hi| is 0 or 1 (lo is allowed the extra element).
        if len(self.lo) > len(self.hi) + 1:
            heapq.heappush(self.hi, -heapq.heappop(self.lo))
        elif len(self.hi) > len(self.lo):
            heapq.heappush(self.lo, -heapq.heappop(self.hi))

    def median(self):
        if len(self.lo) > len(self.hi):
            return float(-self.lo[0])          # odd count: the extra sits in lo
        return (-self.lo[0] + self.hi[0]) / 2  # even count: straddle the gap


# Proof: the streaming median must match statistics.median recomputed on the
# full prefix after *every* insert, for a seeded-random stream.
random.seed(7)
stream = [random.randint(-50, 50) for _ in range(2_000)]

mf = MedianFinder()
seen = []
for x in stream:
    mf.add(x)
    seen.append(x)
    assert mf.median() == statistics.median(seen)   # vs sorted-recompute

print(f"matched statistics.median on all {len(stream)} prefixes")
print("final running median:", mf.median())
print("heap sizes (lo, hi):", len(mf.lo), len(mf.hi))  # differ by <= 1


matched statistics.median on all 2000 prefixes
final running median: -2.0
heap sizes (lo, hi): 1000 1000


The **two-heaps** idea generalizes anywhere you must track a value at a moving boundary: a *sliding-window median* adds lazy deletion to evict the element that left the window, and the same split powers running percentiles. It's the streaming cousin of the partial-sort tools from **top-K & k-way merge** — instead of the k extremes, you keep the data balanced around its center.

## 7. When to use what

| You need... | Reach for | Why |
|---|---|---|
| Repeatedly pull the min/max while still inserting | `heapq` on a `list` | O(log n) push/pop, O(1) peek |
| The k smallest / largest of n items | `heapq.nsmallest` / `nlargest` | O(n log k), no full sort |
| A priority queue (Dijkstra, A*, scheduling) | `heapq` with `(priority, count, item)` | stable, handles ties, never compares items |
| To merge sorted streams lazily | `heapq.merge` | low-memory k-way merge |
| A thread-safe priority queue | `queue.PriorityQueue` | locking wrapper around `heapq` |
| A one-shot full sort | `sorted()` | a heap isn't faster; only use one if you extract incrementally |

## 8. Complexity cheat-sheet

| Operation | Complexity | Notes |
|---|:---:|---|
| Peek min — `h[0]` | O(1) | min-heap keeps it at the front |
| `heappush` | O(log n) | sift up one root-to-leaf path |
| `heappop` | O(log n) | sift down one path |
| `heappushpop` / `heapreplace` | O(log n) | a single sift, not two |
| `heapify(list)` | **O(n)** | bottom-up build (Floyd) |
| `nsmallest` / `nlargest(k)` | O(n log k) | partial sort |
| Search for an arbitrary value | O(n) | a heap orders parent↔child only, not siblings |
| Build by n separate pushes | O(n log n) | prefer `heapify` |